### First few cells is just data preparation

My implementation is build on top of nequip and allegro python libraries to simplify data load and preprocessing.
The wigner simbols are from e3nn
Here are the links
https://github.com/mir-group/nequip
https://github.com/mir-group/allegro

In [1]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS
import os
from nequip.model import model_from_config


default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cuda:0',
    default_dtype="float64",
    model_dtype="float32",
    allow_tf32=True,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)

os.environ['NEQUIP_NUM_TASKS'] = '16'
# All default_config keys are valid / requested
#_GLOBAL_ALL_ASKED_FOR_KEYS.update(default_config.keys())

In [2]:
config = Config.from_file('./configs/example_ETN_opt.yaml', defaults=default_config)
    

dataset = dataset_from_config(config, prefix="dataset")

validation_dataset = None

dataset[0]

AtomicData(atom_types=[21, 1], cell=[3, 3], edge_cell_shift=[364, 3], edge_index=[2, 364], forces=[21, 3], pbc=[3], pos=[21, 3], total_energy=[1])

In [3]:
# Trainer
from nequip.train.trainer import Trainer
from e3nn import o3

trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

# Some hyperparameteres
Nc = 10 # number of chennels for F features from ETN paper
N_rank_spec = 4 # hidden rank of reduction for type radial tensor
config['Nc'] = Nc
config['N_rank_spec'] = N_rank_spec

# ETN parameters
config['d'] = 4 # dimention of the tensor train
config['N_rank_ett'] = [4, 4, 4] # ranks of tensor train



# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train
)

DEBUG:root:* Initialize Output
  ...generate file name results/aspirin/example/log
  ...open log file results/aspirin/example/log
  ...generate file name results/aspirin/example/metrics_epoch.csv
  ...open log file results/aspirin/example/metrics_epoch.csv
  ...generate file name results/aspirin/example/metrics_initialization.csv
  ...open log file results/aspirin/example/metrics_initialization.csv
  ...generate file name results/aspirin/example/metrics_batch_train.csv
  ...open log file results/aspirin/example/metrics_batch_train.csv
  ...generate file name results/aspirin/example/metrics_batch_val.csv
  ...open log file results/aspirin/example/metrics_batch_val.csv
  ...generate file name results/aspirin/example/best_model.pth
  ...generate file name results/aspirin/example/last_model.pth
  ...generate file name results/aspirin/example/trainer.pth
  ...generate file name results/aspirin/example/config.yaml
Torch device: cuda:0
instantiate Loss
...Loss_param = dict(
...   optional_arg

In [4]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
from torch import nn
import math

trainer.model = final_model

# Test configuration stores as dict of parameters
data0 = AtomicData.to_AtomicDataDict(dataset[0])

In [30]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
    
from torch import nn
import math

# forward pass
data_new = final_model(data0)



### Main module

I'll skip the implementation of feature vector F because it heavily relies on nequip code for neighbor lists operations.
If you want you can look into the following files or ask me to add it to the notebook.

Otherwise look into this file

allegro/modules/ETN.py 

and corresponding layers


In [31]:
from typing import Optional
import math
from e3nn.util.codegen import CodeGenMixin
from torch import fx

import torch
from torch_runstats.scatter import scatter

from nequip.data import AtomicDataDict
from nequip.nn import GraphModuleMixin

from allegro import _keys

from torch import nn
from e3nn import o3


class EdgeFeatures_F(nn.Module, GraphModuleMixin):
    def __init__(self,
                 num_types: int,
                 Nc: int, 
                 num_basis: int = 8, 
                 N_rank_spec: int = 4,
                 irreps_in=None,
                 out_field: str = _keys.EDGE_FEATURES_F):
        
        super().__init__()
        self.out_field = out_field
        
        self.irreps_edge_sh = irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]
        
        # set up irreps
        self._init_irreps(
            irreps_in=irreps_in,
            required_irreps_in=[
                AtomicDataDict.EDGE_ATTRS_KEY,
                AtomicDataDict.EDGE_EMBEDDING_KEY,
                _keys.EDGE_TYPE_KEY
            ],
            irreps_out={out_field: o3.Irreps([(Nc, ir) for _, ir in self.irreps_edge_sh])}
        )
        

        
        # parameters of the system
        self.num_types = num_types # number of types
        
        
        # Parameters of the network
        self.Nc = Nc # number of output features
        self.num_basis = num_basis # number of radial basis functions
        self.N_rank_spec = N_rank_spec # encoding of species after one_hot
        
        # tensors for atomic features encoding
        lmax = self.irreps_edge_sh.lmax # maximum spherical harmonic
        
        self._module = EdgeFeatures_FFunction(
            lmax=lmax,
            num_types=self.num_types,
            Nc=self.Nc,
            num_basis=self.num_basis,
            N_rank_spec=self.N_rank_spec,
            irreps_edge_sh=self.irreps_edge_sh,
        )
        

    def forward(self, data: AtomicDataDict.Type) -> AtomicDataDict.Type:
        data[self.out_field] = self._module(data[AtomicDataDict.EDGE_EMBEDDING_KEY],
                                            data[_keys.EDGE_TYPE_KEY],
                                            data[AtomicDataDict.EDGE_ATTRS_KEY])
        
        return data


class EdgeFeatures_FFunction(CodeGenMixin, torch.nn.Module):
    """Module implementing an MLP according to provided options."""

    in_features: int
    out_features: int

    def __init__(
        self,
         lmax: int,
         num_types: int,
         Nc: int, 
         num_basis: int = 8, 
         N_rank_spec: int = 4,
         irreps_edge_sh: o3.Irreps = o3.Irreps("1x0e + 1x1o + 1x2e")
    ):
        super().__init__()

        
        # Code
        params = {}
        graph = fx.Graph()
        tracer = fx.proxy.GraphAppendingTracer(graph)

        def Proxy(n):
            return fx.Proxy(n, tracer=tracer)

        Q = Proxy(graph.placeholder("x"))
        atom_types_embed = Proxy(graph.placeholder("z"))
        Y = Proxy(graph.placeholder("y"))
        norm_from_last: float = 1.0

        base = torch.nn.Module()

        # make weights
        A = torch.empty(lmax + 1, N_rank_spec, num_types**2)
        A.normal_()
        
        B =  torch.empty(lmax + 1, Nc, num_basis, N_rank_spec)           
        B.normal_()

        # generate code
        params[f"A"] = A
        A = Proxy(graph.get_attr(f"A"))

        params[f"B"] = B
        B = Proxy(graph.get_attr(f"B"))


        # Algo from ETN paper to gen F (notation preserved)
        a = A[:, :, atom_types_embed].squeeze(-1)
        b = torch.einsum('Lrnk,LkE,En->ELr', B, a, Q)
    
        F = torch.concat([torch.einsum('Em,En->Emn', Y[:, slices],
                                       b[:, l]) for l, slices in enumerate(irreps_edge_sh.slices())], dim = -2)

        graph.output(F.node)

        for pname, p in params.items():
            setattr(base, pname, torch.nn.Parameter(p))


        self._codegen_register({"_forward": fx.GraphModule(base, graph)})

    def forward(self, x, z, y):
        return self._forward(x, z, y)

### Usage Example

In [43]:
torch.manual_seed(250)

features_F = EdgeFeatures_F(num_types= config['num_types'],
                            Nc = config['Nc'],
                            N_rank_spec = config['N_rank_spec'],
                            irreps_in = final_model.irreps_out,
                            out_field = _keys.EDGE_FEATURES_F)


/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(


In [44]:
data_new_new = features_F(data_new)

In [45]:
data_new_new['atomic_energy'][0]

tensor([-618.1436], grad_fn=<SelectBackward0>)

In [46]:
data_new_new['atomic_energy'][0]

tensor([-618.1436], grad_fn=<SelectBackward0>)

### Some equivariance testing of the final model

In [47]:

import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
from torch import nn
import math

data = data0


import copy

data_rot = {key: torch.clone(data0[key]) for key in data0}

irreps_sh = o3.Irreps('1x0e + 1x1o + 1x2e') #o3.Irreps.spherical_harmonics(lmax=2)
irreps_sh_r = o3.Irreps('1x1o')

alpha, beta, gamma = o3.rand_angles(100)

rot_matrix = irreps_sh.D_from_angles(alpha[0], beta[0], gamma[0])
rot_matrix_r = irreps_sh_r.D_from_angles(alpha[0], beta[0], gamma[0])


data_rot['pos'] = data_rot['pos'] @ rot_matrix_r

In [48]:
torch.manual_seed(32)

data_out = final_model(data)

data_out_rot = final_model(data_rot)


F = data_out['node_features_F'] # features
F_rot =data_out_rot['node_features_F'] # features from rotated positions
F_rot_rot = torch.einsum('Njn,jk->Nkn', F_rot, rot_matrix.T) # rotated features from rotated positions

assert torch.allclose(F, F_rot_rot, atol=1e-05)
print('F is equivariant')
    
    
ETN_out = data_out['node_features_ETN'] # ETN out features
ETN_out_rot =data_out_rot['node_features_ETN'] # ETN out features from rotated positions
ETN_out_rot_rot = torch.einsum('Njn,jk->Nkn', ETN_out_rot, rot_matrix.T) # rotated ETN out features from rotated positions

assert torch.allclose(ETN_out, ETN_out_rot_rot, atol=1e-05)
print('ETN forward output is equivariant')

at_en = data_out['atomic_energy'] # atomic energy
at_en_rot = data_out_rot['atomic_energy'] # atomic energy from rotated positions

assert torch.allclose(at_en, at_en_rot, atol=1e-05)
print('atomic energy is invariant')

tot_en = data_out['total_energy'] # atomic energy
tot_en_rot = data_out_rot['total_energy'] # atomic energy from rotated positions

assert torch.allclose(tot_en, tot_en_rot, atol=1e-05)
print('total energy is invariant')


f_out = data_out['forces'] # atomic forces
f_out_rot =data_out_rot['forces'] # atomic forces from rotated positions
f_out_rot_rot = f_out_rot @ rot_matrix_r.T # rotated forces from rotated positions

assert torch.allclose(f_out, f_out_rot_rot, atol=1e-05)
print('Output forces are equivariant')

F is equivariant
ETN forward output is equivariant
atomic energy is invariant
total energy is invariant
Output forces are equivariant


In [13]:
torch.manual_seed(32)

data_out = ETN(final_model(data))

data_out_rot = ETN(final_model(data_rot))


F = data_out['node_features_F'] # features
F_rot =data_out_rot['node_features_F'] # features from rotated positions
F_rot_rot = torch.einsum('Njn,jk->Nkn', F_rot, rot_matrix.T) # rotated features from rotated positions

assert torch.allclose(F, F_rot_rot, atol=1e-05)
print('F is equivariant')
    
    
ETN_out = data_out['node_features_ETN'] # ETN out features
ETN_out_rot =data_out_rot['node_features_ETN'] # ETN out features from rotated positions
ETN_out_rot_rot = torch.einsum('Njn,jk->Nkn', ETN_out_rot, rot_matrix.T) # rotated ETN out features from rotated positions

assert torch.allclose(ETN_out, ETN_out_rot_rot, atol=1e-05)
print('ETN forward output is equivariant')

at_en = data_out['atomic_energy'] # atomic energy
at_en_rot = data_out_rot['atomic_energy'] # atomic energy from rotated positions

assert torch.allclose(at_en, at_en_rot, atol=1e-05)
print('atomic energy is invariant')

tot_en = data_out['total_energy'] # atomic energy
tot_en_rot = data_out_rot['total_energy'] # atomic energy from rotated positions

assert torch.allclose(tot_en, tot_en_rot, atol=1e-05)
print('total energy is invariant')


f_out = data_out['forces'] # atomic forces
f_out_rot =data_out_rot['forces'] # atomic forces from rotated positions
f_out_rot_rot = f_out_rot @ rot_matrix_r.T # rotated forces from rotated positions

assert torch.allclose(f_out, f_out_rot_rot, atol=1e-05)
print('Output forces are equivariant')

F is equivariant
ETN forward output is equivariant
atomic energy is invariant
total energy is invariant
Output forces are equivariant
